# Lesson 1.3 - Resource Group And Network Module

**Level: Beginner**

## Learning Objectives
- Support both new and existing resource groups.
- Use the AVM VNet module to create networking.
- Connect Terraform data flow across locals, variables, and modules.

**Project file(s) produced:** `terraform/foundation/resource_group.tf`, `terraform/foundation/network.tf`

## Concepts
This project can create a new resource group or reuse a sandbox one.

The network file uses Azure Verified Modules (AVM) for consistent, tested VNet creation.

In [ ]:
%%writefile terraform/foundation/resource_group.tf
locals {
  create_resource_group = var.resource_group_name == null
}

resource "azurerm_resource_group" "this" {
  count = local.create_resource_group ? 1 : 0

  name     = local.names.resource_group
  location = var.location
  tags     = var.tags
}

data "azurerm_resource_group" "existing" {
  count = local.create_resource_group ? 0 : 1
  name  = var.resource_group_name
}

locals {
  resource_group_name = local.create_resource_group ? azurerm_resource_group.this[0].name : data.azurerm_resource_group.existing[0].name
  resource_group_id   = local.create_resource_group ? azurerm_resource_group.this[0].id : data.azurerm_resource_group.existing[0].id
  rg_location         = local.create_resource_group ? azurerm_resource_group.this[0].location : data.azurerm_resource_group.existing[0].location
}

In [ ]:
%%writefile terraform/foundation/network.tf
module "vnet" {
  source  = "Azure/avm-res-network-virtualnetwork/azurerm"
  version = "0.18.0"

  name          = local.names.vnet
  parent_id     = local.resource_group_id
  location      = local.rg_location
  address_space = var.vnet_address_space

  subnets = {
    for key, subnet in var.subnets :
    key => {
      name             = local.subnet_names[key]
      address_prefixes = subnet.address_prefixes
    }
  }

  tags = var.tags
}

## Exercise
Make the module create an additional subnet for private endpoints and then reference it in a comment for future AKS add-ons.

In [ ]:
# TODO: Add another subnet key in variables.tf and keep this network module unchanged.
# TODO: Run terraform plan and verify the second subnet appears.

print('Validation command: terraform -chdir=terraform/foundation plan')

---
**Next:** [Lesson 2.1 - AKS Cluster And GPU Node Pool](../phase-2-terraform-platform/lesson-2.1-aks-cluster-and-gpu-node-pool.ipynb)